In [172]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from os import getenv
import pandas as pd
from pandas import DataFrame
from datasets import load_dataset, Dataset, DatasetDict
import asyncio

In [ ]:
# Load environment variables and dataset
load_dotenv()
ds: DatasetDict = load_dataset("cais/hle")
ds = ds["test"]

In [ ]:
ds = ds.filter(lambda row: row["image_preview"]==None and row["rationale_image"]==None)


Filter: 100%|██████████| 2500/2500 [00:05<00:00, 446.34 examples/s]


In [121]:
df: DataFrame = ds.to_pandas()
df_noimg = df.drop(["image", "image_preview", "rationale_image"], axis=1)

In [123]:
min_count = df_noimg["category"].value_counts().min()

In [128]:
df_noimg_balanced = df.groupby("category", group_keys=False).apply(lambda x: x.sample(min_count, random_state=42))

C:\Users\jasha\AppData\Local\Temp\ipykernel_4076\971522531.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_noimg_balanced = df.groupby("category", group_keys=False).apply(lambda x: x.sample(min_count, random_state=42))


In [129]:
df_noimg_balanced["category"].value_counts()

category
Biology/Medicine             57
Chemistry                    57
Computer Science/AI          57
Engineering                  57
Humanities/Social Science    57
Math                         57
Other                        57
Physics                      57
Name: count, dtype: int64

In [133]:
llm=ChatOpenAI(base_url=getenv("OPENROUTER_BASE_URL"), api_key=getenv("OPENROUTER_API_KEY"), model="openai/gpt-4.1")

In [186]:
template = """Create a description of this question that is free of details 
that reveal the exact question content, such a terms, numbers, and people.
Make it very general. Respond with just the description.
 
{question}
"""
questions = [template.format(question=q) for q in df_noimg_balanced["question"]]
async def get_descriptions(questions):
    responses = await llm.abatch(questions, config={"max_concurrency": 40})
    print(responses[0])
    return [res.content for res in responses]
df_noimg_balanced["description"] = await get_descriptions(questions)
# def generate_description(row):
#     global askllm
#     if askllm == True:
#         print(llm.invoke("""Create a description of this question that is free of details 
#                that reveal the exact question content, such a terms, numbers, and people.
#                Make it very general. Respond with just the description.
               
#                {question}
#                """.format(question=row["question"])).content)
#         askllm = False
#     return row["question"]

content='This is a multiple-choice question that asks about the type of relationship observed between certain biological markers and imaging-based assessment scales in a specific medical condition affecting newborns.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 144, 'total_tokens': 177, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-4.1', 'system_fingerprint': None, 'id': 'gen-1760397414-nBnfBxMNyO0pabNUokKI', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--32d8cb25-43f2-436e-ae79-e3391a733461-0' usage_metadata={'input_tokens': 144, 'output_tokens': 33, 'total_tokens': 177, 'input_token_details': {}, 'output_token_details': {}}


In [188]:
df_noimg_balanced.to_parquet("HLE with descriptions.parquet")